In [ ]:
from py123d.api import SceneFilter

In [ ]:
from py123d.api import get_filtered_scenes

scene_filter = SceneFilter(
    datasets=["nuplan-mini"],
    split_names=None,
    log_names=None,
    scene_uuids=None,
    target_iteration_duration_s=0.1,  # 10Hz iteration frequency
    future_duration_s=10.0,  # Look up to 1 second into the future.
    history_duration_s=5.0,  # Look up to 0.5 seconds into the past.
    # required_scene_modalities=["ego_state_se3", "lidar.lidar_merged"],
    required_scene_modalities=["ego_state_se3"],
)
scenes = get_filtered_scenes(scene_filter)

dataset_splits = set(scene.log_metadata.split for scene in scenes)
print(f"Found {len(scenes)} scenes from {len(dataset_splits)} datasplits:")
for split in dataset_splits:
    print(f" - {split}")

In [ ]:
# Init PDM-Closed

from nav123d.pdm.pdm_closed_planner import get_pdm_closed_planner

pdm_closed_planner = get_pdm_closed_planner()

In [ ]:
# Scan all scenes to find Boston/Singapore reproducers.
# Records: per-scene map, outcome (OK / FAR / CRASH), distance of trajectory[0] from ego,
# and centerline diagnostics. This is how we get a deterministic reproducer instead of
# relying on np.random.choice(scenes).

from collections import defaultdict

import numpy as np

from nav123d.pdm.pdm_closed_planner import PDMClosedInput, get_pdm_closed_planner

scan_iteration = 0
scan_results = []  # list of dicts, one per scene

for scene_idx, scene in enumerate(scenes):
    map_name = scene.log_metadata.map_name if hasattr(scene.log_metadata, "map_name") else "<unknown>"

    try:
        modality = scene.get_custom_modality_at_iteration(scan_iteration, "scenario")
        lane_group_ids = [int(id_) for id_ in modality.data["route_roadblock_ids"]]
        ego_state_se3 = scene.get_ego_state_se3_at_iteration(scan_iteration)
        box_detections_se3 = scene.get_box_detections_se3_at_iteration(scan_iteration)
        traffic_light_detections = scene.get_traffic_light_detections_at_iteration(scan_iteration)
        planner_input = PDMClosedInput(
            ego_state_se2=ego_state_se3.ego_state_se2,
            box_detections_se2=box_detections_se3.box_detections_se2,
            traffic_light_detections=traffic_light_detections,
        )
        map_api = scene.get_map_api()

        planner = get_pdm_closed_planner()  # fresh planner per scene (avoid cached centerline)
        planner.initialize(map_api, lane_group_ids)
        trajectory = planner.compute_planner_trajectory(planner_input)

        traj_xy = trajectory.pose_se2_array[:, :2]
        ego_xy = ego_state_se3.ego_state_se2.rear_axle_se2.array[:2]
        start_offset = float(np.linalg.norm(traj_xy[0] - ego_xy))
        max_offset = float(np.linalg.norm(traj_xy - ego_xy, axis=1).max())

        centerline_xy = planner._centerline.array[:, :2]
        centerline_finite = bool(np.isfinite(centerline_xy).all())
        gaps = np.linalg.norm(np.diff(centerline_xy, axis=0), axis=1) if len(centerline_xy) > 1 else np.array([0.0])
        centerline_start_offset = float(np.linalg.norm(centerline_xy[0] - ego_xy))

        outcome = "OK" if start_offset < 5.0 else "FAR"
        scan_results.append(
            {
                "idx": scene_idx,
                "map": map_name,
                "outcome": outcome,
                "start_offset_m": start_offset,
                "max_offset_m": max_offset,
                "centerline_start_offset_m": centerline_start_offset,
                "centerline_n_points": int(centerline_xy.shape[0]),
                "centerline_finite": centerline_finite,
                "centerline_min_gap_m": float(gaps.min()),
                "centerline_max_gap_m": float(gaps.max()),
                "err": None,
            }
        )

    except Exception as e:
        scan_results.append(
            {
                "idx": scene_idx,
                "map": map_name,
                "outcome": "CRASH",
                "start_offset_m": float("nan"),
                "max_offset_m": float("nan"),
                "centerline_start_offset_m": float("nan"),
                "centerline_n_points": -1,
                "centerline_finite": False,
                "centerline_min_gap_m": float("nan"),
                "centerline_max_gap_m": float("nan"),
                "err": f"{type(e).__name__}: {e}",
            }
        )

# Summary by map
by_map = defaultdict(lambda: defaultdict(int))
for r in scan_results:
    by_map[r["map"]][r["outcome"]] += 1

print(f"\nScan over {len(scan_results)} scenes:\n")
for map_name in sorted(by_map):
    counts = dict(by_map[map_name])
    print(f"  {map_name:35s} {counts}")

print("\nFailing / suspicious scenes:")
for r in scan_results:
    if r["outcome"] != "OK":
        print(
            f"  idx={r['idx']:3d} map={r['map']:30s} outcome={r['outcome']:5s} "
            f"start_off={r['start_offset_m']:8.2f}m  "
            f"cl_start_off={r['centerline_start_offset_m']:8.2f}m  "
            f"cl_n={r['centerline_n_points']:4d}  "
            f"cl_min_gap={r['centerline_min_gap_m']:.4f}m  "
            f"err={r['err']}"
        )

In [ ]:
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
from py123d.visualization.matplotlib.observation import add_scene_on_ax

# Deterministic scene selection. Set to None to use the first failing scene from
# the scan; set to an int to pin a specific scene by index.
TARGET_SCENE_IDX: Optional[int] = None

if TARGET_SCENE_IDX is None:
    try:
        failing = [r for r in scan_results if r["outcome"] != "OK"]
        TARGET_SCENE_IDX = failing[0]["idx"] if failing else 0
    except NameError:
        TARGET_SCENE_IDX = 0
    print(f"Auto-selected scene idx={TARGET_SCENE_IDX} (first failing or 0)")

scene = scenes[TARGET_SCENE_IDX]
iteration = 0


modality = scene.get_custom_modality_at_iteration(iteration, "scenario")
assert modality is not None, "Scenario modality not found at iteration 2."
lane_group_ids = [int(id_) for id_ in modality.data["route_roadblock_ids"]]
print(f"map={getattr(scene.log_metadata, 'map_name', '?')}  first_lane_group_id={lane_group_ids[0]}")

fig, ax = plt.subplots(figsize=(10, 10))
add_scene_on_ax(ax, scene, iteration, radius=100.0, ids=lane_group_ids)

In [ ]:
from nav123d.pdm.pdm_closed_planner import PDMClosedInput

ego_state_se3 = scene.get_ego_state_se3_at_iteration(iteration)
box_detections_se3 = scene.get_box_detections_se3_at_iteration(iteration)
traffic_light_detections = scene.get_traffic_light_detections_at_iteration(iteration)


assert ego_state_se3 is not None, "Ego state modality not found at iteration."
assert box_detections_se3 is not None, "Box detections modality not found at iteration."
assert traffic_light_detections is not None, "Traffic light detections modality not found at iteration."

planner_input = PDMClosedInput(
    ego_state_se2=ego_state_se3.ego_state_se2,
    box_detections_se2=box_detections_se3.box_detections_se2,
    traffic_light_detections=traffic_light_detections,
)

In [ ]:
map_api = scene.get_map_api()
assert map_api is not None, "MapAPI is not available in the scene."
pdm_closed_planner.initialize(map_api, lane_group_ids)
trajectory = pdm_closed_planner.compute_planner_trajectory(planner_input)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

add_scene_on_ax(ax, scene, iteration, radius=100.0, ids=lane_group_ids)

trajectory_se2 = trajectory.pose_se2_array
centerline_se2 = pdm_closed_planner._centerline.array

ax.scatter(trajectory_se2[:, 0], trajectory_se2[:, 1], s=5, color="red", label="PDM-Closed Trajectory", zorder=100)
# ax.plot(centerline_se2[:, 0], centerline_se2[:, 1], linewidth=50, color="blue", label="Centerline", zorder=50)


ax.set_title("PDM-Closed Trajectory Visualization")

In [ ]:
from py123d.geometry.transform import abs_to_rel_se2_array

traj_rel = abs_to_rel_se2_array(ego_state_se3.imu_se2, trajectory_se2)

plt.scatter(traj_rel[:, 0], traj_rel[:, 1], s=5, color="blue", label="PDM-Closed Trajectory (Relative)", zorder=100)